# DentalGPT panoramic pipeline (Kaggle runner)

One image goes through three in-distribution steps, all with question shapes taken from the DentalGPT paper:

```text
panoramic X-ray
   -> 14 presence questions (Figure 7 wording, A. True / B. False), whole image (kept as a separate result)
   -> PRESENCE_LEVEL="region": region by region, the same question for every one of the 14 findings,
      whatever the whole image answered
      (REGION_PROMPT="words": "... present in the upper right quadrant of this image", whole image;
       REGION_PROMPT="crop": the verbatim question on the region crop) -> present if any region says A; region set
   -> COUNT_LEVEL="region": as soon as a region says A for a countable finding, "How many teeth in the upper
      right quadrant ..." there; the finding's count is the sum
      COUNT_LEVEL="overall": one whole-image tooth-count question per positive countable finding (Figure 9 wording)
   -> JSON per image, deterministic dentist summary, TP/FP/TN/FN evaluation, region counts, side check
   -> location truth: ground-truth boxes are translated into the same quadrants by a vision LLM
      (numbered boxes drawn on the image -> FDI quadrant x anterior/posterior), by DentalGPT itself
      (experimental, two multiple-choice questions per box), or by fixed windows
   -> dentist report: a text LLM (the REPORTER role) gets the findings of one image as one dense JSON (every finding,
      region and count with an explicit status) and returns a classified report in the dentist's language
      (REPORT_LANGUAGE); the reply is verified against the data, corrected once if needed, rendered to Markdown
```

Regions are the two jaws or the four FDI quadrants (patient-side names). CELL 3 defines the provider endpoints and Kaggle-secret keys once, then assigns provider/model pairs to the analyzer, adapter and reporter roles. Run the cells in order. Edit only **CELL 3**. A first run executes a short probe that
decides whether the checkpoint needs the `<think>/<answer>` suffix.

In [ ]:
# ============================================================
# CELL 1 - Python dependencies (llama.cpp is compiled later with CUDA)
# ============================================================
%pip install -q "huggingface_hub>=0.26" "openai>=1.55" "requests>=2.31" "pillow>=10.0" "pandas>=1.5"
print("Python dependencies installed.")

In [ ]:
# ============================================================
# CELL 2 - Locate and import the project files
# ============================================================
import os, sys, json, shutil, subprocess
from pathlib import Path

PROJECT_DIR = Path("/kaggle/working/dental_x-ray")  # folder holding the project .py files
REQUIRED_PROJECT_FILES = {"dental_pipeline.py", "dental_eval.py", "llama_runtime.py", "location_adapter.py",
                          "llm_api.py", "report_writer.py"}
if not PROJECT_DIR.is_dir() and Path.cwd().joinpath("dental_pipeline.py").is_file():
    PROJECT_DIR = Path.cwd()
missing = [f for f in REQUIRED_PROJECT_FILES if not (PROJECT_DIR / f).is_file()]
if missing:
    raise FileNotFoundError(f"Missing project files in {PROJECT_DIR}: {missing}")
os.chdir(PROJECT_DIR)
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

import dental_pipeline as dp
import dental_eval as ev
import location_adapter as la
import llm_api
import report_writer as rw
from llama_runtime import LlamaCppServer, build_llama_cpp, download_dentalgpt
print("PROJECT_DIR =", PROJECT_DIR)
print("Project imports succeeded.")

In [ ]:
# ============================================================
# CELL 3 - CONFIGURATION (the only cell to edit between experiments)
# ============================================================
OUTPUT_DIR = "/kaggle/working/dental_outputs"

# Provider endpoints and credentials: define each provider once. Keys are read from environment variables or
# Kaggle Secrets (Add-ons > Secrets); missing keys are allowed until that provider is assigned to a role.
PROVIDERS = {
    "openai": {"base_url": "https://api.openai.com/v1",
               "api_key": ""},
    "openrouter": {"base_url": "https://openrouter.ai/api/v1",
                   "api_key": ""},
    "nvidia": {"base_url": "https://integrate.api.nvidia.com/v1",
               "api_key": llm_api.secret("NVIDIA_API_KEY", required=False)},
    "gemini": {"base_url": "https://generativelanguage.googleapis.com/v1beta/openai/",
               "api_key": llm_api.secret("GEMINI_API_KEY", required=False)},
}
llm_api.configure_providers(PROVIDERS)

# Assign one provider/model to each API role. Model-specific request options stay with the role.
ANALYZER = {"provider": "openrouter", "model": "qwen/qwen3-vl-235b-a22b-thinking",
            "request_options": {"extra_body": {"provider": {"only": ["novita"],
                                                            "allow_fallbacks": False}}}}
ADAPTER = {"provider": "nvidia", "model": "google/gemma-4-31b-it",
           "token_param": "max_completion_tokens", "temperature": None,
           "max_output_tokens": 8192, "max_boxes_per_call": 12}
# Dentist report (CELL 15): a text model turns the findings of one image into a classified report in the dentist's
# language. It never sees the image, so any strong text model will do (here the adapter's). Extra keys:
# "max_output_tokens" and "repairs" (correction turns when the reply fails verification; default 1).
REPORTER = {"provider": "nvidia", "model": "google/gemma-4-31b-it",
            "token_param": "max_completion_tokens", "temperature": None, "max_output_tokens": 8192}
REPORT_LANGUAGE = "English"  # the language every sentence of the report is written in, e.g. "Persian", "German"
REPORT_IMAGES = None         # None = every image with a result; N = only the first N (sorted by id)

# Model backend: local DentalGPT through llama.cpp, or ANALYZER above for a hosted comparison.
BACKEND = "api"  # "local" | "api"

# Local DentalGPT files (Q8_0 if VRAM allows; never Q4_K_M for reported numbers).
HF_REPO_ID = "mradermacher/DentalGPT-7B-1026-GGUF"
MODEL_FILENAME = "DentalGPT-7B-1026.Q6_K.gguf"
MMPROJ_FILENAME = "DentalGPT-7B-1026.mmproj-f16.gguf"
MODEL_DIR = "/kaggle/working/models/dentalgpt"
HF_TOKEN_SECRET = "HF_TOKEN"  # optional (the GGUF repo is public): environment variable or Kaggle secret

# llama.cpp runtime. Context must hold image tokens + question + answer.
LLAMA_CPP_DIR = "/kaggle/working/llama.cpp"
LLAMA_CPP_REF = "b10516"
SERVER_HOST, SERVER_PORT, SERVER_ALIAS = "127.0.0.1", 8080, "dentalgpt"
SERVER_LOG_PATH = "/kaggle/working/llama_dentalgpt_server.log"
N_GPU_LAYERS = 999
CTX_SIZE = 16384
IMAGE_MAX_TOKENS = 6144   # a 2455x1383 panoramic needs ~4400 tokens; llama.cpp would otherwise cap at 4096
IMAGE_MIN_TOKENS = None   # no floor: crops of small panoramics stay at native size, as in training
CUDA_ARCH = None          # None = auto-detect (P100 fallback 60)
BUILD_JOBS = 4
SERVER_STARTUP_TIMEOUT = 300.0

# Generation: one greedy attempt per question.
MAX_TOKENS = 4096
TEMPERATURE = 0.0
CACHE_PROMPT = True       # reuse the image KV prefix across the questions of one image
REQUEST_TIMEOUT_SECONDS = 600.0

# Protocol: the two levels and how a region is put to the model (README, "Two levels").
MODE = "auto"               # "auto" = probe decides; or force "plain" / "tagged"
PRESENCE_LEVEL = "region"   # "overall": whole image only | "region": every region asked about all 14 findings (whole image kept separately)
COUNT_LEVEL = "region"      # "overall": one whole-image count per positive countable finding | "region": a count in each region answering A
REGION_SCHEME = "quadrant"  # "quadrant" (UR, UL, LL, LR) | "arch" (upper, lower)
REGION_PROMPT = "words"     # "words": region named in the question, whole image sent | "crop": verbatim question on the crop
PROTOCOL = dp.Protocol(presence_level=PRESENCE_LEVEL, count_level=COUNT_LEVEL,
                       region_scheme=REGION_SCHEME, region_prompt=REGION_PROMPT)
PROBE_IMAGES = 2

# Location truth: how ground-truth boxes are translated into the region windows DentalGPT is scored on.
#   "llm"      - numbered boxes drawn on the radiograph, a strong vision LLM (OpenAI-compatible API) returns
#                FDI quadrant x anterior/posterior per box; mapped onto the windows in Python. Recommended.
#   "fdm"      - DentalGPT itself, two Figure-7-shaped multiple-choice questions per box drawn in red
#                (experimental; needs BACKEND="local" and the probe's RUN_MODE)
#   "geometry" - fixed crop windows, no model calls (the previous behaviour)
EVALUATE_LOCATION = True  # False: skip location scoring and truth adapter; keep inference and total-count scoring
LOCATION_TRUTH = "llm"
# LOCATION_TRUTH="llm" uses the ADAPTER role configured above.

# Datasets to run and score. kind "yolo": UMFIH 14-class layout; kind "dentex": DENTEX split.
# "limit" runs only the first N images (sorted by id); None = all.
DATASETS = [
    {"name": "umfih_test", "kind": "yolo", "limit": 2,
     "images": "/kaggle/working/umfih_14class/data/test/images",
     "labels": "/kaggle/working/umfih_14class/data/test/labels"},
    # {"name": "umfih_external", "kind": "yolo", "limit": None,
    #  "images": "/kaggle/working/umfih_14class/external/images",
    #  "labels": "/kaggle/working/umfih_14class/external/labels"},
    # DENTEX: use the fully labeled train split (705 images, COCO-style JSON) and/or the 50-image
    # validation split (validation_triple.json); DentalGPT never trained on DENTEX, so both are held-out.
    # The 250-image DENTEX test split ships as raw LabelMe files without an official class mapping; skip it.
    # {"name": "dentex_train", "kind": "dentex", "limit": None,
    #  "images": "/kaggle/working/DENTEX/training_data/quadrant-enumeration-disease/xrays",
    #  "annotations": "/kaggle/working/DENTEX/training_data/quadrant-enumeration-disease/train_quadrant_enumeration_disease.json"},
    # {"name": "dentex_val", "kind": "dentex", "limit": None,
    #  "images": "/kaggle/working/DENTEX/validation_data/quadrant_enumeration_disease/xrays",
    #  "annotations": "/kaggle/working/DENTEX/validation_triple.json"},
]

NEEDS_LOCAL_RUNTIME = BACKEND == "local"
print("backend =", BACKEND, "| mode =", MODE, "| protocol =", PROTOCOL, "| location truth =", LOCATION_TRUTH,
      "| report =", REPORTER["model"], "in", REPORT_LANGUAGE, "| datasets =", [d["name"] for d in DATASETS])

In [ ]:
# ============================================================
# CELL 4 - Environment diagnostics
# ============================================================
import platform
print("Python:", platform.python_version(), "|", platform.platform())


def detect_cuda_arch(fallback="60"):
    try:
        output = subprocess.check_output(["nvidia-smi", "--query-gpu=compute_cap", "--format=csv,noheader"],
                                         text=True, stderr=subprocess.STDOUT)
        arch = output.strip().splitlines()[0].strip().replace(".", "")
        if arch.isdigit():
            return arch
    except Exception as exc:
        print("CUDA architecture auto-detection failed:", exc)
    print(f"Falling back to CUDA architecture {fallback}.")
    return fallback


CUDA_ARCH_RESOLVED = None
if NEEDS_LOCAL_RUNTIME:
    for executable in ("git", "cmake", "nvcc", "nvidia-smi"):
        print(f"{executable:12s}:", shutil.which(executable))
    if shutil.which("nvidia-smi"):
        subprocess.run(["nvidia-smi"], check=False)
    for executable in ("cmake", "git", "nvcc"):
        if not shutil.which(executable):
            raise RuntimeError(f"{executable} is required to build llama.cpp; enable a GPU accelerator.")
    CUDA_ARCH_RESOLVED = str(CUDA_ARCH) if CUDA_ARCH else detect_cuda_arch("60")
    print("CUDA_ARCH_RESOLVED =", CUDA_ARCH_RESOLVED)
else:
    print("API backend selected; local checks skipped.")

In [ ]:
# ============================================================
# CELL 5 - Build or find the pinned llama.cpp server
# ============================================================
LLAMA_SERVER = None
if NEEDS_LOCAL_RUNTIME:
    # Reuses an existing build only if it was built from LLAMA_CPP_REF; otherwise rebuilds.
    LLAMA_SERVER = Path(build_llama_cpp(source_dir=LLAMA_CPP_DIR, cuda_arch=CUDA_ARCH_RESOLVED,
                                        jobs=BUILD_JOBS, ref=LLAMA_CPP_REF)).resolve()
    print("llama-server =", LLAMA_SERVER)
else:
    print("API backend selected; llama.cpp build skipped.")

In [ ]:
# ============================================================
# CELL 6 - Download the DentalGPT GGUF and its vision projector
# ============================================================
MODEL_PATH = MMPROJ_PATH = None
if NEEDS_LOCAL_RUNTIME:
    files = download_dentalgpt(model_dir=MODEL_DIR, repo_id=HF_REPO_ID, model_filename=MODEL_FILENAME,
                               mmproj_filename=MMPROJ_FILENAME,
                               hf_token=llm_api.secret(HF_TOKEN_SECRET, required=False))
    MODEL_PATH, MMPROJ_PATH = Path(files.model_path).resolve(), Path(files.mmproj_path).resolve()
    for label, path in (("Language model", MODEL_PATH), ("Vision projector", MMPROJ_PATH)):
        print(f"{label}: {path} ({path.stat().st_size / 1024**3:.2f} GiB)")
else:
    print("API backend selected; download skipped.")

In [ ]:
# ============================================================
# CELL 7 - Start a clean DentalGPT llama.cpp server
# ============================================================
import requests

previous = globals().get("server")
server = None
if NEEDS_LOCAL_RUNTIME:
    if previous is not None:
        try:
            previous.stop()
        except Exception as exc:
            print("Previous server cleanup:", exc)
    server = LlamaCppServer(binary=LLAMA_SERVER, model_path=MODEL_PATH, mmproj_path=MMPROJ_PATH,
                            host=SERVER_HOST, port=SERVER_PORT, alias=SERVER_ALIAS, n_gpu_layers=N_GPU_LAYERS,
                            ctx_size=CTX_SIZE, image_max_tokens=IMAGE_MAX_TOKENS, image_min_tokens=IMAGE_MIN_TOKENS,
                            startup_timeout=SERVER_STARTUP_TIMEOUT, log_path=SERVER_LOG_PATH)
    server.start(reuse_existing=False)
    ids = [m.get("id") for m in requests.get(f"{server.base_url}/v1/models", timeout=10).json().get("data", [])]
    if SERVER_ALIAS not in ids:
        raise RuntimeError(f"Expected alias {SERVER_ALIAS!r}; /v1/models returned {ids}. See {SERVER_LOG_PATH}.")
    print("DentalGPT server verified at", server.base_url)
else:
    print("API backend selected; server startup skipped.")

In [ ]:
# ============================================================
# CELL 8 - Model runner (one image + one question -> one answer)
# ============================================================
if NEEDS_LOCAL_RUNTIME:
    runner = dp.VisionRunner(base_url=f"{server.base_url}/v1", model=SERVER_ALIAS, max_tokens=MAX_TOKENS,
                             temperature=TEMPERATURE, timeout=REQUEST_TIMEOUT_SECONDS, local=True,
                             cache_prompt=CACHE_PROMPT)
else:
    runner = dp.VisionRunner.from_api(ANALYZER, max_tokens=MAX_TOKENS, temperature=TEMPERATURE,
                                      timeout=REQUEST_TIMEOUT_SECONDS)
print("Runner ready:", runner.settings())

In [ ]:
# ============================================================
# CELL 9 - Load ground truth for every dataset (no model calls)
# ============================================================
GT = {}
for spec in DATASETS:
    if spec["kind"] == "yolo":
        gt = ev.load_yolo(spec["images"], spec["labels"])
    elif spec["kind"] == "dentex":
        gt = ev.load_dentex(spec["images"], spec["annotations"])
    else:
        raise ValueError(f"unknown dataset kind {spec['kind']!r}")
    if spec.get("limit"):
        gt = dict(sorted(gt.items())[: spec["limit"]])
    GT[spec["name"]] = gt
    positives = sum(1 for g in gt.values() if g["boxes"])
    print(f"{spec['name']}: {len(gt)} images, {positives} with at least one finding, "
          f"{sum(len(g['boxes']) for g in gt.values())} boxes")

In [ ]:
# ============================================================
# CELL 10 - Probe: does this checkpoint need the <think>/<answer> suffix?
# ============================================================
# Sends the bare presence and count questions, with and without the suffix, on a few images.
# "plain" is chosen when the model already reasons in tags on its own; "tagged" only when the
# suffix produces the tagged format. A mode already recorded in a run manifest or a saved
# probe is reused, so restarting the notebook never flips the mode of a resumed run.
RUN_MODE = MODE
if MODE == "auto" and not NEEDS_LOCAL_RUNTIME:
    RUN_MODE = "plain"  # the suffix is a DentalGPT training artifact; a hosted model gets the bare questions
    print("API backend: mode 'plain', probe skipped (set MODE to force 'tagged').")
elif MODE == "auto":
    manifests = [Path(OUTPUT_DIR) / d["name"] / "manifest.json" for d in DATASETS]
    recorded = next((m for m in manifests if m.is_file()), None)
    probe_path = Path(OUTPUT_DIR, "probe.json")
    if recorded is not None:
        RUN_MODE = json.loads(recorded.read_text(encoding="utf-8"))["mode"]
        print(f"Reusing mode {RUN_MODE!r} recorded in {recorded}; probe skipped.")
    elif probe_path.is_file():
        PROBE = json.loads(probe_path.read_text(encoding="utf-8"))
        RUN_MODE = PROBE["recommended_mode"]
        print(f"Reusing saved probe {probe_path}.")
    else:
        first = next(iter(GT.values()))
        probe_paths = [g["path"] for _, g in sorted(first.items())[:PROBE_IMAGES]]
        PROBE = dp.probe(runner, probe_paths, n=PROBE_IMAGES)
        probe_path.parent.mkdir(parents=True, exist_ok=True)
        probe_path.write_text(json.dumps(PROBE, indent=1), encoding="utf-8")
        RUN_MODE = PROBE["recommended_mode"]
if "PROBE" in globals() and MODE == "auto":
    for mode in dp.MODES:
        s = PROBE[mode]
        print(f"{mode:7s}: think tags {s['think_tag_rate']:.0%} | answer tags {s['answer_tag_rate']:.0%} | "
              f"parsed {s['parse_rate']:.0%} | truncated {s['truncation_rate']:.0%}")
    print("\nSample raw answers (plain mode):")
    for row in PROBE["plain"]["samples"][:2]:
        print(f"--- {row['kind']} | parsed={row['parsed']}\n{row['text'][:600]}")
if RUN_MODE not in dp.MODES:
    raise ValueError(f"RUN_MODE must be one of {dp.MODES}, got {RUN_MODE!r}")
print("\nRUN_MODE =", RUN_MODE)

In [ ]:
# ============================================================
# CELL 11 - Run every dataset (resumable: finished images are skipped)
# ============================================================
# Provenance is hashed into the run manifest, so a changed checkpoint or image budget cannot be
# silently mixed into a resumed run.
PROVENANCE = ({"model_file": MODEL_FILENAME, "mmproj_file": MMPROJ_FILENAME, "llama_cpp_ref": LLAMA_CPP_REF,
               "ctx_size": CTX_SIZE, "image_max_tokens": IMAGE_MAX_TOKENS, "image_min_tokens": IMAGE_MIN_TOKENS}
              if NEEDS_LOCAL_RUNTIME else llm_api.public(ANALYZER))
RUN_DIRS = {}
for spec in DATASETS:
    images = {image_id: g["path"] for image_id, g in GT[spec["name"]].items()}
    RUN_DIRS[spec["name"]] = dp.run_dataset(runner, images, Path(OUTPUT_DIR) / spec["name"],
                                            mode=RUN_MODE, protocol=PROTOCOL, resume=True, provenance=PROVENANCE)
    print("Saved:", RUN_DIRS[spec["name"]])

In [ ]:
# ============================================================
# CELL 12 - Location truth: translate ground-truth boxes into the region windows (resumable)
# ============================================================
# Runs once per dataset and adapter (independent of the model run; can be re-used across modes and levels).
# One JSON per image under <dataset>/location_truth/<adapter>/boxes, drawn images under .../drawn for audit.
# Boxes the adapter cannot place fall back to the fixed windows and are counted as such.
ADAPTED = {}
if not EVALUATE_LOCATION:
    adapter = None
elif LOCATION_TRUTH == "llm":
    adapter = la.LLMAdapter.from_api(ADAPTER, timeout=REQUEST_TIMEOUT_SECONDS)
elif LOCATION_TRUTH == "fdm":
    if not NEEDS_LOCAL_RUNTIME:
        raise RuntimeError("LOCATION_TRUTH='fdm' needs the local DentalGPT server (BACKEND='local').")
    adapter = la.FdmAdapter(runner, mode=RUN_MODE)
elif LOCATION_TRUTH == "geometry":
    adapter = None
else:
    raise ValueError(f"unknown LOCATION_TRUTH {LOCATION_TRUTH!r}")

for spec in DATASETS:
    name = spec["name"]
    if not EVALUATE_LOCATION:
        print(f"{name}: location scoring disabled; skipping location truth adapter")
        continue
    if adapter is None or not PROTOCOL.uses_regions:
        print(f"{name}: location truth from fixed windows (and FDI quadrant labels where the dataset has them)")
        continue
    ADAPTED[name] = la.adapt_dataset(adapter, GT[name], Path(OUTPUT_DIR) / name / "location_truth" / adapter.name, resume=True)
    print(f"{name}: {la.summarize(ADAPTED[name])}")
    agreement = ev.truth_agreement(GT[name], ADAPTED[name])
    if agreement["boxes_with_fdi"]:
        # DENTEX carries FDI quadrant labels: exact windows, so this is the adapter's own accuracy against the fixed windows.
        print(f"{name}: adapter vs FDI truth {agreement}")


In [ ]:
# ============================================================
# CELL 13 - Evaluate: TP/FP/TN/FN per finding (regional and whole-image), counts, regions, side check
# ============================================================
import pandas as pd
from IPython.display import display

REPORTS = {}
for spec in DATASETS:
    name = spec["name"]
    results = dp.load_results(Path(OUTPUT_DIR) / name)
    if not results:
        print(f"{name}: no results under {Path(OUTPUT_DIR) / name}; run CELL 11 first.")
        continue
    truth = ev.apply_adapted(GT[name], ADAPTED[name]) if EVALUATE_LOCATION and name in ADAPTED else GT[name]
    REPORTS[name] = ev.evaluate(truth, results, dataset=name, out_dir=Path(OUTPUT_DIR) / name / "evaluation",
                                evaluate_location=EVALUATE_LOCATION)
    print(f"\n===== {name} =====")
    display(pd.DataFrame([REPORTS[name]["summary"]]).T)
    display(pd.DataFrame(REPORTS[name]["presence"]).set_index("condition"))
    if REPORTS[name]["whole_image"]:
        print("whole-image answers alone (what the regional pass recovered, and what it cost):")
        display(pd.DataFrame(REPORTS[name]["whole_image"]).set_index("condition"))
    if REPORTS[name]["counts"]:
        display(pd.DataFrame(REPORTS[name]["counts"]).set_index("condition"))
    if REPORTS[name]["region_counts"]:
        display(pd.DataFrame(REPORTS[name]["region_counts"]).set_index(["condition", "region"]))
    if REPORTS[name]["regions"]:
        display(pd.DataFrame(REPORTS[name]["regions"]).set_index("condition"))
    if "side_agreement" in REPORTS[name]["summary"]:
        # Quadrant windows are fixed to the image (UR, LR = image left). Far above 50%: the model reads the
        # quadrant words as patient sides, as dental_pipeline.QUADRANT_WORDS_ARE_PATIENT_SIDE assumes; far
        # below: flip that constant (the words move to the mirrored windows) and rerun.
        print("side agreement:", REPORTS[name]["summary"]["side_agreement"])
if len(REPORTS) > 1:
    print("\n===== pooled over shared findings =====")
    display(pd.DataFrame(ev.pooled_presence(list(REPORTS.values()))).set_index("condition"))
print("Presence rows use only findings the dataset annotates. Unparseable answers are excluded from TP/FP/TN/FN and "
      "reported as counts; the per-image complete-case rate and recall are strict and count them as not caught. "
      "summary.location_truth says which method placed the true boxes for the region rows. In the regions table, "
      "pred_all_regions_rate far above truth_all_regions_rate means the model answered A in every region whenever "
      "the whole image was positive, i.e. it ignored the region.")

In [ ]:
# ============================================================
# CELL 14 - Inspect one image: dentist summary and raw model answers
# ============================================================
INSPECT_DATASET = DATASETS[0]["name"]
INSPECT_IMAGE = None  # None = first image of the dataset
SHOW_RAW_CALLS = False

results = dp.load_results(Path(OUTPUT_DIR) / INSPECT_DATASET)
image_id = INSPECT_IMAGE or next(iter(sorted(results)))
result = results[image_id]
print(dp.dentist_report(result))
truth = [b["condition"] for b in GT[INSPECT_DATASET][image_id]["boxes"]]
print("\nGround-truth boxes:", {c: truth.count(c) for c in dict.fromkeys(truth)} or "none")
if INSPECT_DATASET in ADAPTED:
    for k, record in enumerate(ADAPTED[INSPECT_DATASET][image_id]["boxes"], 1):
        print(f"  box {k}: {record['condition']} -> {record['regions']} ({record['source']}; fixed windows {record['geometry']})")
if SHOW_RAW_CALLS:
    for call in result["calls"]:
        print(f"\n--- {call['stage']} | {call['condition']} | {call['region']} | finish={call['finish_reason']}")
        print(call["text"][:800])

In [ ]:
# ============================================================
# CELL 15 - Dentist report: one LLM call per image over the structured findings (resumable)
# ============================================================
# The analyzer answered up to 14 + R x 14 (+ counts) narrow questions per image. The report writer gets them as
# one dense JSON (every finding, every region, every count, each with an explicit status), returns a report in
# REPORT_LANGUAGE as JSON (one entry per finding in seven sections, impression, caveats), which is verified against
# the data (every finding exactly once, statuses unchanged, nothing invented), sent back once for correction if it
# fails, and rendered to Markdown. One .json + one .md per image under <dataset>/reports/<model>-<language>/reports;
# a reply that fails twice keeps the deterministic dentist summary, marked as such. The model never sees the image.
from IPython.display import Markdown, display

ANALYZER_NAME = MODEL_FILENAME if NEEDS_LOCAL_RUNTIME else ANALYZER["model"]
writer = rw.ReportWriter.from_api(REPORTER, language=REPORT_LANGUAGE, timeout=REQUEST_TIMEOUT_SECONDS)
print("Report writer:", writer.public())
WRITTEN = {}
for spec in DATASETS:
    name = spec["name"]
    results = dp.load_results(Path(OUTPUT_DIR) / name)
    if not results:
        print(f"{name}: no results under {Path(OUTPUT_DIR) / name}; run CELL 11 first.")
        continue
    WRITTEN[name] = rw.report_dataset(writer, results, Path(OUTPUT_DIR) / name / "reports" / writer.run_name,
                                      analyzer=ANALYZER_NAME, resume=True, limit=REPORT_IMAGES)
    print(f"{name}: {rw.summarize_reports(WRITTEN[name])}")

# One report to read (INSPECT_DATASET / INSPECT_IMAGE from CELL 14 when set, else the first image).
reports = WRITTEN.get(globals().get("INSPECT_DATASET")) or next(iter(WRITTEN.values()), {})
if reports:
    wanted = globals().get("INSPECT_IMAGE")
    shown = reports[wanted] if wanted in reports else reports[next(iter(sorted(reports)))]
    print(f"{shown['image_id']}: verified={shown['verified']} | attempts={len(shown['attempts'])} | problems={shown['problems']}")
    display(Markdown(shown["markdown"]))